# 3.b Industrial Anomaly Detection on MVTec AD

**Section 3: Autoencoders and Anomaly Detection**

Implement Industrial Equipment Anomaly Detection on the MVTec AD Kaggle
dataset using Autoencoder and  build the model by using the following steps:

Steps:
• Preprocess images (resize, normalize) for input into a convolutional autoencoder.
• Build a convolutional autoencoder (CNN-based encoder-decoder) to learn the
representations of normal object images.
• Use reconstruction errors in pixel values to detect anomalies in test images.
Dataset: MVTec AD is an anomaly detection dataset for industrial defect detection,
containing images of normal and defective objects.

Goal: Detect defects in objects by training an image-based autoencoder on images of
defect-free objects.


**Newly written code.** Exercise 3.b on the source page has only the problem statement. The cells below were written for this repository, in the same style as the other autoencoder exercises: a convolutional encoder-decoder, trained only on defect-free images, with anomalies flagged when the per-image MSE exceeds the 95th percentile of the training error.

Place one MVTec AD category in a folder named `mvtec_ad` in the working directory:

- `mvtec_ad/train/good/` — defect-free images used for training
- `mvtec_ad/test/good/` — defect-free test images (label 0)
- `mvtec_ad/test/` subfolders other than `good` — defective test images (label 1)

Images are resized to 128×128 and scaled to [0, 1]. 128 is divisible by 8, so three pooling stages line up with three upsampling stages.


In [ ]:
# 1. Import libraries
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import load_img, img_to_array


In [ ]:
# 2. Load and preprocess images
# One MVTec category laid out as:
#   mvtec_ad/train/good/
#   mvtec_ad/test/good/
#   mvtec_ad/test/<defect type>/
dataset_path = 'mvtec_ad'
image_size = (128, 128)


def load_images_from_dir(folder):
    images = []
    valid = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        if not os.path.isfile(path):
            continue
        if not name.lower().endswith(valid):
            continue
        img = load_img(path, color_mode='rgb', target_size=image_size)
        images.append(img_to_array(img) / 255.0)

    if len(images) == 0:
        raise ValueError('No images found in ' + folder)

    return np.array(images)


# Normal (defect-free) images used for training
X_normal = load_images_from_dir(os.path.join(dataset_path, 'train', 'good'))

# Test images: good = normal (0), any other subfolder = anomaly (1)
X_test_list = []
y_test_list = []

test_root = os.path.join(dataset_path, 'test')

for defect_type in sorted(os.listdir(test_root)):
    folder = os.path.join(test_root, defect_type)
    if not os.path.isdir(folder):
        continue

    images = load_images_from_dir(folder)
    label = 0 if defect_type == 'good' else 1

    X_test_list.append(images)
    y_test_list.append(np.full(len(images), label, dtype='int32'))

if len(X_test_list) == 0:
    raise ValueError('No test images found in ' + test_root)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

print('Normal training images:', X_normal.shape)
print('Test images:', X_test.shape)


In [ ]:
# 3. Define convolutional autoencoder
model = keras.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Encoder
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2), padding='same'),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2), padding='same'),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2), padding='same'),

    # Decoder
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.UpSampling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.UpSampling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.UpSampling2D((2, 2)),

    layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')
])


In [ ]:
# 4. Compile model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)


In [ ]:
# 5. Train only on defect-free images
history = model.fit(
    X_normal, X_normal,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)


In [ ]:
# 6. Calculate reconstruction error
train_predictions = model.predict(X_normal)
test_predictions = model.predict(X_test)

train_error = np.mean(
    np.square(X_normal - train_predictions),
    axis=(1, 2, 3)
)

test_error = np.mean(
    np.square(X_test - test_predictions),
    axis=(1, 2, 3)
)


In [ ]:
# 7. Set threshold
threshold = np.percentile(train_error, 95)

print('Threshold:', threshold)


In [ ]:
# 8. Detect anomalies
y_pred = (test_error > threshold).astype(int)

print('Detected Anomalies:', np.sum(y_pred))

print('Classification Report:')
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=['Normal', 'Anomaly'],
    zero_division=0
))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


In [ ]:
# 9. Plot reconstruction error
plt.hist(train_error, bins=50, alpha=0.6, label='Normal')
plt.hist(test_error, bins=50, alpha=0.6, label='Test')
plt.axvline(threshold, color='red', linestyle='dashed',
            label='Threshold')

plt.title('MVTec AD Anomaly Detection')
plt.xlabel('Reconstruction Error')
plt.ylabel('Number of Images')
plt.legend()
plt.show()


In [ ]:
# 10. Plot training loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Convolutional Autoencoder Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
